In [ ]:
#| hide
from pullup import *


# pullup

Get a project from your laptop to running somewhere: what kind of project it is, the steps that
release it, the GitHub Actions that do it on a server, and the machines and DNS underneath.

pullup is a library, not a CLI. It computes and runs; whatever you put in front of it draws.
Every object answers with plain dicts, so a web panel, a terminal UI and a test all read the same
thing.

## Install

```sh
pip install pullup            # the project, the plan, the runner, and reading workflows
pip install "pullup[cloud]"   # + GitHub, secrets, servers and DNS
```

The cloud half is imported where it is used. A pipeline that runs `pytest` needs none of it.

## What kind of project is this?

```python
from pullup import Project

p = Project('~/code/myrepo')
p.kind           # 'nbdev' | 'fastship' | 'maturin' | 'crate' | 'plain'
p.steps()        # the commands that release a project of that kind
```

`kind` is read from the folder: `settings.ini` or `[tool.nbdev]` makes it nbdev, `Cargo.toml`
with a maturin backend makes it maturin, and so on. Rust is asked before fastship, because a
maturin crate has a `pyproject.toml` too and the fastship flow would try to upload a wheel that
only the CI runners can build.

## Running the steps

```python
from pullup import Release

r = Release('~/code/myrepo', python='~/code/myrepo/.venv/bin/python')
await r.start()          # run the first unfinished step, then keep going while they pass
r.state()                # every step, its status, exit code, timings, and unset keys
r.tail(80)               # the transcript, escapes gone
r.stop()                 # kill the running step; what already passed stays passed
```

- **`start(step_id='', auto=True)`** runs one step and, while `auto`, the next after it passes.
  It stops at the first failure, because a release that publishes from a tree whose tests never
  passed is worse than one that stops.
- **`skip(id)`** marks a step done without running it. **`reset()`** forgets the results and
  leaves the commands alone.
- **`save(steps)`** persists an edited plan and refuses one with no runnable command in it.
  **`reset_plan()`** goes back to the default for this kind of project.
- **`needs()`** lists every environment key the plan names; **`missing(step)`** says which of
  them nothing can produce, so a panel can grey out a step before anyone presses it.

Commands run on a pty through [ptymini](https://pypi.org/project/ptymini/), so a step that asks
for a password gets one: `write(data)` sends keystrokes and `resize(cols, rows)` follows the
window. `subscribe()` hands you a queue of terminal frames primed with everything that already
scrolled past, so a viewer that joins in the middle of a release sees the whole run.

Where the project is a uv project, steps go through `uv run`, which syncs the lock first — so a
venv that has drifted is repaired rather than failing halfway through with an import error.

## Where things are written

Nothing is written outside the directory you name:

```python
Release(root, dir='.pullup')     # the default
Release(root, dir='.myapp')      # your own
```

## The other pipelines

`Deploy` and `Drive` are the same object with different defaults. `Deploy` knows the deploy
settings schema and generates the workflow that runs it. `Drive` is the deployment written out as
library calls you can run one at a time — build the image, wrap it in Caddy, open a tunnel, add a
hostname — each carrying the code it would run, what it needs first, and what it produces, so you
can read it before you run it.

## Workflows

```python
from pullup import Workflows, levels

w = Workflows('~/code/myrepo')
w.parsed()               # every workflow: triggers, jobs, and the rows to draw them in
w.status()               # whether the GitHub half is usable, and what is missing if not
w.runs(limit=20)         # recent runs, newest first
w.dispatch('ci.yml')     # start one; GitHub refuses unless it declares workflow_dispatch
```

`levels(jobs)` arranges jobs into rows: everything that can start now, then what follows, by
longest path. A job that needs one the workflow never defines still gets a row, and so does a
cycle — a workflow you cannot look at is worse than one drawn oddly.

Reading a workflow needs nothing but pullup. Where gheasy is installed its parser is used instead,
so a file gheasy wrote round-trips through the parser it was written with.

## Secrets and the environment

```python
from pullup import EnvStore

env = EnvStore()                 # through dockeasy: keychain first, then the env file
env.get('GITHUB_TOKEN')
env.values(['GITHUB_TOKEN', 'TWINE_PASSWORD'])
```

A pipeline takes one as `env=` and puts what it holds into each step's environment. `venv_env`
builds that environment from a project's interpreter, and strips a frozen host's `PYTHONHOME` and
`PYTHONPATH` on the way — a child that keeps them imports the bundle's standard library under
another interpreter and dies somewhere that names nothing to do with the cause.

## Which of your packages raised

```python
from pullup import blame

blame(r.tail(400), checkouts=['~/code/gheasy'], family=['gheasy', 'nbdev'])
```

The deepest frame belonging to one of your own packages, the error line, and where to open it —
preferring your checkout over the copy in site-packages, because editing site-packages is editing
something the next install discards.

## Servers and DNS

`Infra` reaches Hetzner through vpseasy and Cloudflare through cfeasy, using the tokens the
environment store holds. Part of `pullup[cloud]`.